Lets implement RNN

In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"

df = pd.read_csv(
    url,
    sep=";",
    na_values="?",
    low_memory=False
)

print(df.head())
print(df.shape)

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  
(2075259, 9)


In [3]:
df = df[['Date', 'Time', 'Global_active_power']]
df.head()

,Date,Time,Global_active_power
0,16/12/2006,17:24:00,4.216
1,16/12/2006,17:25:00,5.360
2,16/12/2006,17:26:00,5.374
3,16/12/2006,17:27:00,5.388
4,16/12/2006,17:28:00,3.666


In [4]:
df["Global_active_power"] = pd.to_numeric(df["Global_active_power"], errors="coerce")
df = df.dropna()
df.shape

(2049280, 3)

In [5]:
data = df[:50000]
data.shape

(50000, 3)

In [6]:
s = int(len(data) * 0.8)

train_data = data[:s]
test_data = data[s:]

train_data.shape, test_data.shape

((40000, 3), (10000, 3))

In [7]:
from torch.utils.data import Dataset

In [8]:
class hpc_dataset(Dataset):
    def __init__(self, data, seq_length, scaler=None, is_train=True):
        self.seq_length = seq_length
        
        if is_train:
            self.scaler = StandardScaler()
            scaled_data = self.scaler.fit_transform(np.array(data).reshape(-1, 1)).flatten()
        else:
            if scaler is None:
                raise ValueError("Must pass the fitted training scaler when is_train=False")
            self.scaler = scaler
            scaled_data = self.scaler.transform(np.array(data).reshape(-1, 1)).flatten()
            
        self.X, self.y = self.create_sequences(scaled_data, seq_length)
        
    def create_sequences(self, data, seq_length):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i : i + seq_length])
            y.append(data[i + seq_length])
        return np.array(X), np.array(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        features = torch.tensor(self.X[idx], dtype=torch.float32).unsqueeze(-1)
        label = torch.tensor(self.y[idx], dtype=torch.float32).unsqueeze(-1)
        
        return features, label

In [9]:
SEQ_LEN = 30

train_column = train_data['Global_active_power'].values
train_dataset = hpc_dataset(train_column, seq_length=SEQ_LEN, is_train=True)

fitted_scaler = train_dataset.scaler

test_column = test_data['Global_active_power'].values
test_dataset = hpc_dataset(test_column, seq_length=SEQ_LEN, scaler = fitted_scaler, is_train=False)

In [10]:
X, y = train_dataset[0]
X, y

(tensor([[1.8673],
         [2.7138],
         [2.7242],
         [2.7346],
         [1.4603],
         [1.3523],
         [1.4870],
         [1.4855],
         [1.4618],
         [1.4574],
         [2.0390],
         [2.7523],
         [2.6132],
         [2.6458],
         [1.7474],
         [1.2517],
         [1.1673],
         [1.2857],
         [1.1644],
         [1.5062],
         [3.1090],
         [4.4498],
         [3.9466],
         [2.5762],
         [2.0582],
         [1.1510],
         [1.1422],
         [1.1362],
         [1.1584],
         [1.0992]]),
 tensor([0.7603]))

In [11]:
x, y = test_dataset[0]
X, y

(tensor([[1.8673],
         [2.7138],
         [2.7242],
         [2.7346],
         [1.4603],
         [1.3523],
         [1.4870],
         [1.4855],
         [1.4618],
         [1.4574],
         [2.0390],
         [2.7523],
         [2.6132],
         [2.6458],
         [1.7474],
         [1.2517],
         [1.1673],
         [1.2857],
         [1.1644],
         [1.5062],
         [3.1090],
         [4.4498],
         [3.9466],
         [2.5762],
         [2.0582],
         [1.1510],
         [1.1422],
         [1.1362],
         [1.1584],
         [1.0992]]),
 tensor([1.1229]))

In [12]:
from torch.utils.data import DataLoader

In [13]:
BATCH_SIZE = 64

train_loader = DataLoader(
    dataset=train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    drop_last=True      
)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,   
    drop_last=False
)

In [14]:
for features, labels in train_loader:
    print(f"Features shape: {features.shape}")
    print(f"Labels shape: {labels.shape}")
    break

Features shape: torch.Size([64, 30, 1])
Labels shape: torch.Size([64, 1])


In [15]:
class myRNN(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.hidden_size = hidden_size

        self.Wxh = torch.nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
        self.Whh = torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.Why = torch.nn.Parameter(torch.randn(hidden_size, output_size) * 0.01)

        self.bh = torch.nn.Parameter(torch.zeros(hidden_size))
        self.by = torch.nn.Parameter(torch.zeros(output_size))

    def forward(self, x):

        batch_size = x.size(0)

        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        for t in range(x.size(1)):

            x_t = x[:, t, :]

            h = torch.tanh( x_t @ self.Wxh + h @ self.Whh + self.bh)

        y_pred = h @ self.Why + self.by

        return y_pred

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [17]:
model = myRNN( input_size = 1, hidden_size = 32, output_size = 1).to(device)

In [18]:
for name, param in model.named_parameters():
    print(name, param.shape)

Wxh torch.Size([1, 32])
Whh torch.Size([32, 32])
Why torch.Size([32, 1])
bh torch.Size([32])
by torch.Size([1])


In [19]:
total_params = sum(
    p.numel() for p in model.parameters()
)

print(total_params)

1121


In [20]:
X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape)
print("y:", y_batch.shape)

X: torch.Size([64, 30, 1])
y: torch.Size([64, 1])


In [21]:
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

y_pred = model(X_batch)

print("prediction:", y_pred.shape)

prediction: torch.Size([64, 1])


In [22]:
criterion = torch.nn.MSELoss()

loss = criterion(y_pred, y_batch)

print("Loss:", loss.item())

Loss: 1.0206581354141235


In [23]:
model.zero_grad()

loss.backward()

print("Whh shape:", model.Whh.shape)
print("Whh grad shape:", model.Whh.grad.shape)
print("Whh grad norm:", model.Whh.grad.norm().item())

Whh shape: torch.Size([32, 32])
Whh grad shape: torch.Size([32, 32])
Whh grad norm: 0.0071467021480202675


In [24]:
for name, param in model.named_parameters():
    print(name)
    print("  shape:", param.shape)
    print("  grad shape:", param.grad.shape)
    print("  grad norm:", param.grad.norm().item())

Wxh
  shape: torch.Size([1, 32])
  grad shape: torch.Size([1, 32])
  grad norm: 0.11887801438570023
Whh
  shape: torch.Size([32, 32])
  grad shape: torch.Size([32, 32])
  grad norm: 0.0071467021480202675
Why
  shape: torch.Size([32, 1])
  grad shape: torch.Size([32, 1])
  grad norm: 0.12618981301784515
bh
  shape: torch.Size([32])
  grad shape: torch.Size([32])
  grad norm: 0.007411113008856773
by
  shape: torch.Size([1])
  grad shape: torch.Size([1])
  grad norm: 0.1260053813457489


In [27]:
def training_loop(model: torch.nn.Module, data: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer, device: torch.device = device):

    model = model.to(device = device)
    model.train()

    loss_log = []

    for X, y in data:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)

        loss = loss_fn(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_log.append(loss.item())

    avg_loss = sum(loss_log) / len(loss_log)

    return avg_loss

In [29]:
def testing_loop(model: torch.nn.Module, data: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, device: torch.device = device):
    model = model.to(device)
    model.eval()
    with torch.inference_mode():
        loss_log = []
        for X, y in data:
            X, y = X.to(device), y.to(device)

            y_pred = model(X)

            loss = loss_fn(y_pred, y)

            loss_log.append(loss.item())

    avg_loss = sum(loss_log) / len(loss_log)
    
    return avg_loss

In [30]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(params = model.parameters(), lr = 0.001)

In [31]:
EPOCHS = 30

for epoch in range(EPOCHS):

    train_loss = training_loop(
        model=model,
        data=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device
    )

    test_loss = testing_loop(
        model=model,
        data=test_loader,
        loss_fn=loss_fn,
        device=device
    )

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | " f"Train Loss: {train_loss:.4f} | " f"Test Loss: {test_loss:.4f}")

Epoch 01/30 | Train Loss: 0.1622 | Test Loss: 0.1307
Epoch 02/30 | Train Loss: 0.0875 | Test Loss: 0.1143
Epoch 03/30 | Train Loss: 0.0853 | Test Loss: 0.1121
Epoch 04/30 | Train Loss: 0.0851 | Test Loss: 0.1109
Epoch 05/30 | Train Loss: 0.0846 | Test Loss: 0.1127
Epoch 06/30 | Train Loss: 0.0853 | Test Loss: 0.1096
Epoch 07/30 | Train Loss: 0.0847 | Test Loss: 0.1126
Epoch 08/30 | Train Loss: 0.0848 | Test Loss: 0.1101
Epoch 09/30 | Train Loss: 0.0844 | Test Loss: 0.1107
Epoch 10/30 | Train Loss: 0.0849 | Test Loss: 0.1090
Epoch 11/30 | Train Loss: 0.0845 | Test Loss: 0.1084
Epoch 12/30 | Train Loss: 0.0849 | Test Loss: 0.1103
Epoch 13/30 | Train Loss: 0.0845 | Test Loss: 0.1103
Epoch 14/30 | Train Loss: 0.0841 | Test Loss: 0.1091
Epoch 15/30 | Train Loss: 0.0844 | Test Loss: 0.1106
Epoch 16/30 | Train Loss: 0.0843 | Test Loss: 0.1117
Epoch 17/30 | Train Loss: 0.0844 | Test Loss: 0.1104
Epoch 18/30 | Train Loss: 0.0847 | Test Loss: 0.1109
Epoch 19/30 | Train Loss: 0.0845 | Test Loss: 

In [32]:
model.state_dict()

OrderedDict([('Wxh',
              tensor([[-0.0838,  0.0835, -0.0905, -0.2481,  0.1058, -0.0962, -0.0971,  0.0718,
                       -0.0719, -0.1018, -0.0692,  0.0505, -0.0770,  0.0614, -0.0920,  0.0767,
                        0.0626,  0.0986, -0.1501, -0.0698, -0.2223,  0.0775,  0.0699, -0.0904,
                       -0.0976,  0.0712, -0.0793, -0.0900, -0.0736,  0.0904,  0.0851,  0.0816]],
                     device='cuda:0')),
             ('Whh',
              tensor([[-0.0728,  0.0550, -0.0066,  ..., -0.0068,  0.0301,  0.0240],
                      [ 0.0051, -0.0273,  0.0358,  ..., -0.0022, -0.0029, -0.0304],
                      [ 0.0104, -0.0028, -0.0008,  ...,  0.0301,  0.0043, -0.0009],
                      ...,
                      [ 0.0272, -0.0078,  0.0291,  ..., -0.0353, -0.0067, -0.0210],
                      [ 0.0115, -0.0162, -0.0064,  ..., -0.0253,  0.0113,  0.0155],
                      [ 0.0007, -0.0096,  0.0035,  ..., -0.0248, -0.0138, -0.0074]],
    

In [33]:
from pathlib import Path

MODEL_PATH = Path("models")

MODEL_PATH.mkdir(parents=True, exist_ok = True)

MODEL_NAME = "simple_RNN_model.pth"

save_path = MODEL_PATH / MODEL_NAME

torch.save(model.state_dict(), save_path)